# Step 3 — Vectorization & FAISS Index

This notebook reads `chunks/chunks.csv`, encodes every chunk into a vector using each of the
three fine-tuned encoders, and saves one FAISS index per model.

**Prerequisite:** `baseline_step1.ipynb` must have been run with weight saving enabled, producing:
```
weights/bert-base-uncased_IHC/   or   weights/bert-base-uncased_ISHate/
weights/hateBERT_IHC/            or   weights/hateBERT_ISHate/
weights/roberta-base_IHC/        or   weights/roberta-base_ISHate/
```

**Outputs:**
```
index/bert-base-uncased.faiss
index/hateBERT.faiss
index/roberta-base.faiss
index/documents.json    ← lookup table: {chunk_id → text}
```

**Key design choices:**
- Encoder: `AutoModel` (base transformer only — classification head from fine-tuning is ignored)
- Embedding: `[CLS]` token — `last_hidden_state[:, 0, :]`
- Similarity: cosine, via L2-normalization + `IndexFlatIP` (exact search)
- Index type: `IndexIDMap(IndexFlatIP)` — FAISS returns `chunk_id` directly, not a positional row index
- Embedding dimension: read from `model.config.hidden_size`, asserted at runtime

## 1. Imports

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Configurations

> **Known limitation — encoder/index mismatch:** `chunks.csv` contains data from both IHC and ISHate,
> but each encoder is fine-tuned on only one of them. This is accepted for now given the small domain
> gap (both are Twitter hate speech). Two cleaner solutions to revisit later:
> - **Joint fine-tuning:** retrain one model on IHC + ISHate combined, build a single unified index.
> - **Separate indexing:** one index per dataset, each encoded by its matching fine-tuned model, then merge top-k results at query time.

In [3]:
def _resolve_project_dirs():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        weights_dir = candidate / "weights"
        rag_dir = candidate / "RAG"
        if weights_dir.exists() and (rag_dir / "index").exists():
            return candidate, rag_dir, weights_dir
        if weights_dir.exists() and (candidate / "index").exists():
            return candidate, candidate, weights_dir
    raise FileNotFoundError(
        "Could not locate the project root with weights/ and RAG/index directories."
    )

PROJECT_ROOT, RAG_DIR, WEIGHTS_DIR = _resolve_project_dirs()
INDEX_DIR = RAG_DIR / "index"

_ALL_MODELS = {
    "bert-base-uncased_IHC":   {"hf_id": "bert-base-uncased",  "weights_path": str(WEIGHTS_DIR / "bert-base-uncased_IHC")},
    "hateBERT_IHC":            {"hf_id": "GroNLP/hateBERT",    "weights_path": str(WEIGHTS_DIR / "hateBERT_IHC")},
    "roberta-base_IHC":        {"hf_id": "roberta-base",       "weights_path": str(WEIGHTS_DIR / "roberta-base_IHC")},
    "bert-base-uncased_IsHate":{"hf_id": "bert-base-uncased",  "weights_path": str(WEIGHTS_DIR / "bert-base-uncased_ISHate")},
    "hateBERT_IsHate":         {"hf_id": "GroNLP/hateBERT",    "weights_path": str(WEIGHTS_DIR / "hateBERT_ISHate")},
    "roberta-base_IsHate":     {"hf_id": "roberta-base",       "weights_path": str(WEIGHTS_DIR / "roberta-base_ISHate")},
}

ACTIVE_MODELS = [
    "bert-base-uncased_IHC",
    "hateBERT_IHC",
    "roberta-base_IHC",
    "bert-base-uncased_IsHate",
    "hateBERT_IsHate",
    "roberta-base_IsHate",
]

MODEL_CONFIGS = {k: _ALL_MODELS[k] for k in ACTIVE_MODELS}

BATCH_SIZE = 64
MAX_LENGTH = 64    # 55 content tokens + [CLS]/[SEP] + margin

## 3. Load Chunks

Load `chunks/chunks.csv`. The `text` column contains the string that will be encoded.
The `chunk_id` column is the row index and will match the FAISS vector position.

In [4]:
# Chunks were filtered in rag_step2_chunking.ipynb using the RoBERTa tokenizer.
# other tokenizer path : chunks_HateBERT_tokenizer_55_tokens_limit
df = pd.read_csv("chunks/chunks_RoBERTa_tokenizer_55_tokens_limit.csv")[["chunk_id", "text"]]
texts     = df["text"].tolist()
chunk_ids = df["chunk_id"].to_numpy(dtype="int64")
print(f"Chunks to index: {len(texts):,}")
df.head()

Chunks to index: 64,796


,chunk_id,text
0,0,[not hate] we need everyone to keep winning . ...
1,1,[not hate] : the hatred spewed by robert spenc...
2,2,[not hate] are antifa boomers ?
3,3,[not hate] #trucons aren't capable of anything...
4,4,[not hate] wow you really caught that . so ha...


## 4. Encoding Function

Takes a list of strings + a loaded model + tokenizer, returns a float32 numpy array of shape
`(N, 768)` where each row is the `[CLS]` embedding of one chunk.

Uses batched inference with `torch.no_grad()`. Moves tensors to GPU if available.

In [5]:
def encode(texts, model, tokenizer, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        cls_vecs = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_vecs.append(cls_vecs)
    return np.vstack(all_vecs).astype("float32")

## 5. Build One FAISS Index Per Model

For each model in `MODEL_CONFIGS`:
1. Determine which path to load from: local `weights_path` if the folder exists, else `hf_fallback`
2. Load tokenizer and model (`AutoModel`, not classification)
3. Encode all chunks (show progress with tqdm)
4. L2-normalize vectors (`faiss.normalize_L2`) for cosine similarity
5. Build `IndexFlatIP(768)` and add all vectors
6. Save index to `index/{model_name}.faiss`
7. Print: total vectors added, index size on disk

## 5A. Inspect Saved Weights

Before encoding, verify two things per model:
1. **Key prefix** — what do the keys in `model.safetensors` actually look like? A mismatch with `AutoModel` would silently leave the model at base HuggingFace weights.
2. **Classifier head** — are classification-head keys (`classifier.*`, `pooler.*`) present in the saved file? They should NOT end up in the final `AutoModel` (confirmed by `unexpected` keys being dropped via `strict=False`), but we want to see them explicitly.

In [6]:
os.makedirs(str(INDEX_DIR), exist_ok=True)

print("Models that will be encoded:")
for model_name, cfg in MODEL_CONFIGS.items():
    print(f"  {model_name}  →  {cfg['weights_path']}")

for model_name, cfg in MODEL_CONFIGS.items():
    weights_path = cfg["weights_path"]
    hf_id        = cfg["hf_id"]
    print(f"\n=== {model_name} ===")

    # Tokenizer from HuggingFace — local tokenizer.json was saved with a newer
    # tokenizers library version and can't be parsed by the installed version.
    tokenizer = AutoTokenizer.from_pretrained(hf_id)

    # Load the full fine-tuned classification model, then take .base_model to
    # strip the classifier head while keeping the correctly loaded encoder weights.
    full_model = AutoModelForSequenceClassification.from_pretrained(weights_path)
    model      = full_model.base_model   # RobertaModel / BertModel — returns last_hidden_state
    model.eval().to(device)

    vectors = encode(texts, model, tokenizer)

    embed_dim = model.config.hidden_size
    assert vectors.shape == (len(texts), embed_dim), \
        f"Unexpected shape {vectors.shape}, expected ({len(texts)}, {embed_dim})"
    print(f"  Vectors shape: {vectors.shape}  ✓")

    faiss.normalize_L2(vectors)

    inner_index = faiss.IndexFlatIP(embed_dim)
    index = faiss.IndexIDMap(inner_index)
    index.add_with_ids(vectors, chunk_ids)

    out_path = INDEX_DIR / f"{model_name}.faiss"
    faiss.write_index(index, str(out_path))
    print(f"  Saved {out_path}  ({index.ntotal} vectors)")

    del full_model, model
    if device.type == "cuda":
        torch.cuda.empty_cache()

Models that will be encoded:
  bert-base-uncased_IHC  →  /scratch/deep_learning/weights/bert-base-uncased_IHC
  hateBERT_IHC  →  /scratch/deep_learning/weights/hateBERT_IHC
  roberta-base_IHC  →  /scratch/deep_learning/weights/roberta-base_IHC
  bert-base-uncased_IsHate  →  /scratch/deep_learning/weights/bert-base-uncased_ISHate
  hateBERT_IsHate  →  /scratch/deep_learning/weights/hateBERT_ISHate
  roberta-base_IsHate  →  /scratch/deep_learning/weights/roberta-base_ISHate

=== bert-base-uncased_IHC ===


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Encoding: 100%|█████████▉| 1008/1013 [00:41<00:00, 24.25it/s]

Encoding: 100%|█████████▉| 1011/1013 [00:41<00:00, 24.52it/s]

Encoding: 100%|██████████| 1013/1013 [00:41<00:00, 24.21it/s]

  Vectors shape: (64796, 768)  ✓


  Saved /scratch/deep_learning/RAG/index/roberta-base_IsHate.faiss  (64796 vectors)


In [7]:
from safetensors.torch import load_file as load_safetensors  # diagnostic only

for model_name, cfg in MODEL_CONFIGS.items():
    safetensors_path = os.path.join(cfg["weights_path"], "model.safetensors")
    if not os.path.isfile(safetensors_path):
        print(f"\n[{model_name}] No model.safetensors found — skipping")
        continue

    saved_keys = list(load_safetensors(safetensors_path).keys())
    head_keys  = [k for k in saved_keys if any(p in k for p in ("classifier", "pooler"))]

    full_model = AutoModelForSequenceClassification.from_pretrained(cfg["weights_path"])
    encoder    = full_model.base_model
    matched    = sum(1 for k in saved_keys if k in full_model.state_dict())

    print(f"\n{'='*60}")
    print(f"Model : {model_name}")
    print(f"  Total keys in safetensors   : {len(saved_keys)}")
    print(f"  Keys matched (full model)   : {matched} / {len(full_model.state_dict())}")
    print(f"  Classifier/pooler keys      : {len(head_keys)}")
    for k in head_keys:
        print(f"    {k}")
    print(f"  base_model type : {type(encoder).__name__}")
    del full_model


Model : bert-base-uncased_IHC
  Total keys in safetensors   : 201
  Keys matched (full model)   : 151 / 201
  Classifier/pooler keys      : 4
    bert.pooler.dense.bias
    bert.pooler.dense.weight
    classifier.bias
    classifier.weight
  base_model type : BertModel

Model : hateBERT_IHC
  Total keys in safetensors   : 201
  Keys matched (full model)   : 151 / 201
  Classifier/pooler keys      : 4
    bert.pooler.dense.bias
    bert.pooler.dense.weight
    classifier.bias
    classifier.weight
  base_model type : BertModel



Model : roberta-base_IHC
  Total keys in safetensors   : 201
  Keys matched (full model)   : 151 / 201
  Classifier/pooler keys      : 4
    classifier.dense.bias
    classifier.dense.weight
    classifier.out_proj.bias
    classifier.out_proj.weight
  base_model type : RobertaModel

Model : bert-base-uncased_IsHate
  Total keys in safetensors   : 201
  Keys matched (full model)   : 151 / 201
  Classifier/pooler keys      : 4
    bert.pooler.dense.bias
    bert.pooler.dense.weight
    classifier.bias
    classifier.weight
  base_model type : BertModel



Model : hateBERT_IsHate
  Total keys in safetensors   : 201
  Keys matched (full model)   : 151 / 201
  Classifier/pooler keys      : 4
    bert.pooler.dense.bias
    bert.pooler.dense.weight
    classifier.bias
    classifier.weight
  base_model type : BertModel

Model : roberta-base_IsHate
  Total keys in safetensors   : 201
  Keys matched (full model)   : 151 / 201
  Classifier/pooler keys      : 4
    classifier.dense.bias
    classifier.dense.weight
    classifier.out_proj.bias
    classifier.out_proj.weight
  base_model type : RobertaModel


## 6. Save Shared Lookup Table

FAISS returns integer positions, not text. We save `documents.json` as a list of strings
so that position `i` in the FAISS index maps to `documents[i]` in the JSON.

This lookup table is **shared across all three indexes** — the chunk order is identical.

In [8]:
# Keyed by str(chunk_id) — JSON keys are always strings
documents = {str(cid): text for cid, text in zip(chunk_ids, texts)}

with open("index/documents.json", "w") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"Saved index/documents.json  ({len(documents):,} entries)")

Saved index/documents.json  (64,796 entries)


## 7. Sanity Check

Load the hateBERT index and run two sample queries to verify the pipeline end-to-end.

> If result counts vary wildly across queries (e.g. sometimes 0, sometimes 10), `SIMILARITY_THRESHOLD` is too strict. A dedicated test notebook will measure this and switch to top-k if needed.

In [9]:
# Similarity threshold used at retrieval time.
# If result counts vary wildly across queries (sometimes 0, sometimes 10),
# the threshold is too strict — a dedicated test notebook will validate this
# and switch to top-k if needed.
SIMILARITY_THRESHOLD = 0.95

_sc_name = "roberta-base_IsHate"
_sc_cfg  = MODEL_CONFIGS[_sc_name]

test_index = faiss.read_index(str(INDEX_DIR / f"{_sc_name}.faiss"))
with open(INDEX_DIR / "documents.json") as f:
    documents = json.load(f)

tokenizer_sc = AutoTokenizer.from_pretrained(_sc_cfg["hf_id"])
full_sc      = AutoModelForSequenceClassification.from_pretrained(_sc_cfg["weights_path"])
model_sc     = full_sc.base_model.eval().to(device)

def query_index(tweet_text, k=5, threshold=SIMILARITY_THRESHOLD):
    vec = encode([tweet_text], model_sc, tokenizer_sc, batch_size=1)
    faiss.normalize_L2(vec)
    scores, ids = test_index.search(vec, k)
    return [
        (documents[str(ids[0][i])], float(scores[0][i]))
        for i in range(k)
        if ids[0][i] != -1 and scores[0][i] >= threshold
    ]

queries = [
    # --- hate ---
    ("HATE",     "jews are now in full control : canadian man goes to jail for posting on white nationalist websites"),
    ("HATE",     "Mexicans have plenty of open space in their countries to develop they do n't need ours ."),
    ("HATE",     "if you want to see what having a black , kenyan white hating muslime in the whitehouse has created"),
    ("HATE",     "i dont like neggers"),
    # --- not hate ---
    ("HATE",     "i dont like tomatoes"),
    ("NOT HATE", "can they breath fire and have scaly skin ? sure they can light a cross on fire but that doesn't count ."),
    ("NOT HATE", "the only thing linder has ever accomplished is inspiring glenn miller to kill a 14 year old white kid"),
    ("NOT HATE", "The media will praise and encourage the non-white establishment while denouncing the White man 's establishments ."),
    ("NOT HATE", "you ever notice how go text them other hoes backwards is i got options bitch so dont play"),
    ("NOT HATE", "Charges in deadly wreck that killed 5 motorcyclists . Crash takes injured woman 's fiancé , mom ."),
]

for label, tweet in queries:
    print(f"\n[{label}] {tweet}")
    hits = query_index(tweet)
    if hits:
        for text, score in hits:
            print(f"  {score:.3f}  {text}")
    else:
        print("  (no results above threshold)")


[HATE] jews are now in full control : canadian man goes to jail for posting on white nationalist websites


Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding: 100%|██████████| 1/1 [00:00<00:00, 122.81it/s]

  0.998  [hate] : they hate you because the jew controlled mainstream media is brainwashing them . #jewishsupremacy
  0.998  [hate] white people in britain resisting propaganda to racially miscegenate  figures show | more :
  0.998  [hate] japan is able to keep clean his country from refugees . they accept positively from nearly . requests . this is worldrecord !
  0.998  [hate] there is no hate . what they want to likely root out is any form of white european identity . they have declared
  0.998  [hate] there is no hate . what they want to doubtlessly root out is any form of white european identity . they have declared

[HATE] Mexicans have plenty of open space in their countries to develop they do n't need ours .


Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

...

Encoding: 100%|██████████| 1/1 [00:00<00:00, 93.94it/s]

  0.996  [not hate] Ok Ignore the one pic. that is n't so good ....
  0.995  [not hate] the fact that who comments on this cube problem is mostly not a cube stan?? yall be creating tweets so you can get rts and like. shut the fuck up if you dont know anything about cube, jesus
  0.995  [not hate] ur the new leader of the alt right mrtaylor i cant handle it anymor good luck sir i made dad send u a check weu
  0.995  [not hate] Sooo @Shanteshanteeee you ain't get my text message hoe? 🤔
  0.995  [not hate] dis why i didn't want you following me you think everything is about u but i mean if the shoe fits go ahead, lace that bitch up & wear it

[NOT HATE] Charges in deadly wreck that killed 5 motorcyclists . Crash takes injured woman 's fiancé , mom .


Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding: 100%|██████████| 1/1 [00:00<00:00, 149.42it/s]

  0.991  [not hate] Read more : Charges in deadly wreck that killed 5 motorcyclists - LancasterOnline.com News Crash takes injured woman 's fiancé , mom - LancasterOnline.com News
  0.990  [not hate] Ariel Laurie , Israeli Soldier , Grabs Toddler in Australia Kidnaps 6 year old boy , says he 's going to put him into car trunk but mother chases him down .
  0.989  [not hate] Thor Soderberg , 43 , was shot and killed at 61st and Racine .
  0.989  [not hate] 32-year-old Jason `` Jay '' Tate was found with Kelsey Sue Stahl 's car .
  0.989  [not hate] -- A man wanted for attempted murder in New York City was killed Friday afternoon in an exchange of gunfire with officers with a fugitive task force who had gone to a Regency Square-area apartment to arrest him .
